# 01 — Exploratory Data Analysis
Notebook ini mengeksplorasi dataset UCI Heart Disease secara menyeluruh:
- Statistik deskriptif
- Distribusi fitur
- Analisis korelasi
- Deteksi missing values & outlier
- Analisis class imbalance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('Libraries loaded ✓')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/heart.csv')

# Rename kolom jika perlu
COL_NAMES = [
    'age','sex','chest_pain_type','resting_bp','cholesterol',
    'fasting_blood_sugar','resting_ecg','max_heart_rate',
    'exercise_angina','st_depression','st_slope',
    'num_major_vessels','thalassemia','target'
]
if df.columns.tolist() == list(range(14)):
    df.columns = COL_NAMES

# Binarize target (UCI: 0=sehat, 1-4=sakit)
if df['target'].nunique() > 2:
    df['target'] = (df['target'] > 0).astype(int)

print(f'Shape: {df.shape}')
df.head()

## 2. Statistik Deskriptif

In [ ]:
print('=== Info ===')
df.info()
print()
print('=== Describe ===')
df.describe().round(2)

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per kolom:')
print(missing[missing > 0] if missing.any() else 'Tidak ada missing values ✓')

## 3. Distribusi Target (Class Balance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count
vc = df['target'].value_counts()
axes[0].bar(['Low Risk (0)', 'High Risk (1)'], vc.values,
            color=['#2E86AB', '#E84855'], alpha=0.85, edgecolor='white')
axes[0].set_title('Distribusi Kelas', fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie
axes[1].pie(vc.values, labels=['Low Risk', 'High Risk'],
            colors=['#2E86AB', '#E84855'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporsi Kelas', fontweight='bold')

plt.suptitle('Class Distribution — UCI Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_class_distribution.png', bbox_inches='tight')
plt.show()

## 4. Distribusi Fitur Numerik

In [ ]:
NUMERIC = ['age','resting_bp','cholesterol','max_heart_rate','st_depression']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(NUMERIC):
    axes[i].hist(df[df['target']==0][col], bins=25, alpha=0.6,
                 color='#2E86AB', label='Low Risk', edgecolor='white')
    axes[i].hist(df[df['target']==1][col], bins=25, alpha=0.6,
                 color='#E84855', label='High Risk', edgecolor='white')
    axes[i].set_title(col.replace('_',' ').title())
    axes[i].legend(fontsize=9)
    axes[i].spines[['top','right']].set_visible(False)

axes[-1].set_visible(False)
plt.suptitle('Distribusi Fitur Numerik per Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_numeric_distributions.png', bbox_inches='tight')
plt.show()

## 5. Distribusi Fitur Kategorikal

In [ ]:
CATEGORICAL = ['sex','chest_pain_type','fasting_blood_sugar',
               'resting_ecg','exercise_angina','st_slope',
               'num_major_vessels','thalassemia']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(CATEGORICAL):
    ct = df.groupby([col, 'target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['#2E86AB','#E84855'],
            alpha=0.85, edgecolor='white', rot=0)
    axes[i].set_title(col.replace('_',' ').title(), fontsize=10)
    axes[i].set_xlabel('')
    axes[i].legend(['Low Risk','High Risk'], fontsize=8)
    axes[i].spines[['top','right']].set_visible(False)

plt.suptitle('Distribusi Fitur Kategorikal per Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_categorical_distributions.png', bbox_inches='tight')
plt.show()

## 6. Heatmap Korelasi

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 9}
)
ax.set_title('Correlation Heatmap — All Features', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../reports/figures/01_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Korelasi tertinggi dengan target
print('\nTop 10 fitur berkorelasi dengan target:')
print(corr['target'].abs().sort_values(ascending=False).head(11))

## 7. Boxplot — Deteksi Outlier

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 5))

for i, col in enumerate(NUMERIC):
    axes[i].boxplot(
        [df[df['target']==0][col].dropna(),
         df[df['target']==1][col].dropna()],
        labels=['Low', 'High'],
        patch_artist=True,
        boxprops=dict(facecolor='#E6F1FB'),
        medianprops=dict(color='#E84855', linewidth=2)
    )
    axes[i].set_title(col.replace('_',' ').title(), fontsize=10)
    axes[i].spines[['top','right']].set_visible(False)

plt.suptitle('Boxplot Fitur Numerik per Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_boxplots.png', bbox_inches='tight')
plt.show()

## 8. Pairplot Fitur Penting

In [ ]:
top_features = ['age', 'max_heart_rate', 'st_depression',
                'cholesterol', 'resting_bp', 'target']

g = sns.pairplot(
    df[top_features], hue='target',
    palette={0: '#2E86AB', 1: '#E84855'},
    plot_kws={'alpha': 0.5, 's': 20},
    diag_kind='kde'
)
g.fig.suptitle('Pairplot — Top Numeric Features', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('../reports/figures/01_pairplot.png', bbox_inches='tight')
plt.show()

## 9. Ringkasan EDA

In [ ]:
print('═'*55)
print('RINGKASAN EDA — UCI Heart Disease Dataset')
print('═'*55)
print(f'Total sampel   : {len(df)}')
print(f'Jumlah fitur   : {len(df.columns)-1}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Kelas 0 (Low)  : {(df.target==0).sum()} ({(df.target==0).mean()*100:.1f}%)')
print(f'Kelas 1 (High) : {(df.target==1).sum()} ({(df.target==1).mean()*100:.1f}%)')
print()
print('Fitur korelasi tertinggi dengan target:')
top = df.corr()['target'].abs().sort_values(ascending=False).drop('target').head(5)
for feat, val in top.items():
    print(f'  {feat:<25} {val:.3f}')
print()
print('Semua plot tersimpan di ../reports/figures/')
print('Lanjut ke 02_preprocessing.ipynb →')